Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
SPDX-License-Identifier: MIT-0

# Catch a regression

**Track 3, optional.** The travel agent's system prompt says what it can do and never says what it must
decline. Ask it for something outside travel and it answers with HTTP 200, a governed model call and a
healthy span tree. Nothing in any error metric moves. You reproduce that, score it with `Builtin.Refusal`
(a refusal is the correct behaviour for this request), then fix the prompt and prove the fix with the same
evaluator.

In [ ]:
%pip install --quiet "boto3==1.43.126" "botocore==1.43.126"
import boto3

assert tuple(int(x) for x in boto3.__version__.split(".")[:3]) >= (1, 43, 100), (
    f"boto3 too old: {boto3.__version__} (need >=1.43.100 for the cloudwatchomni client)"
)
# If the kernel imported an older botocore before the pip install above, the new
# service models are not loaded yet. Fail here with an actionable message.
services = boto3.session.Session().get_available_services()
assert "cloudwatchomni" in services and "observabilityadmin" in services, (
    "This kernel has no cloudwatchomni client. Restart the kernel (Kernel > Restart) and run this cell again."
)
print(f"boto3 {boto3.__version__} OK - cloudwatchomni and observabilityadmin clients available")

In [ ]:
import json
import os
import pathlib
import time

import boto3

REGION = (
    os.environ.get("AWS_REGION")
    or os.environ.get("AWS_DEFAULT_REGION")
    or boto3.session.Session().region_name
)
if not REGION:
    raise RuntimeError("No AWS region configured. Set AWS_REGION or run `aws configure set region <region>`.")
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

# The CLI pages save identifiers as `export KEY=VALUE` lines in this file, so a
# participant can switch between the CLI and notebook paths at any step.
STATE_FILE = pathlib.Path.home() / ".workshop-omni.env"


def load_state():
    state = {}
    if STATE_FILE.exists():
        for line in STATE_FILE.read_text().splitlines():
            if line.startswith("export ") and "=" in line:
                key, value = line[len("export "):].split("=", 1)
                state[key] = value.strip()
    return state


def save_state(**values):
    state = load_state()
    state.update({k: v for k, v in values.items() if v})
    STATE_FILE.write_text("".join(f"export {k}={v}\n" for k, v in state.items()))
    return state


omni = boto3.client("cloudwatchomni", region_name=REGION)
obs_admin = boto3.client("observabilityadmin", region_name=REGION)
logs = boto3.client("logs", region_name=REGION)
cfn = boto3.client("cloudformation", region_name=REGION)
print(f"Region {REGION}, account ...{ACCOUNT_ID[-4:]}")

In [ ]:
import secrets
import urllib.parse

import requests

ssm = boto3.client("ssm", region_name=REGION)
cognito = boto3.client("cognito-idp", region_name=REGION)


def fast_param(name):
    return ssm.get_parameter(Name=f"/FAST-stack/{name}")["Parameter"]["Value"]


RUNTIME_ARN = fast_param("runtime-arn")
USER_POOL_ID = fast_param("cognito-user-pool-id")
CLIENT_ID = fast_param("cognito-user-pool-client-id")
INVOKE_URL = (f"https://bedrock-agentcore.{REGION}.amazonaws.com/runtimes/"
              f"{urllib.parse.quote(RUNTIME_ARN, safe='')}/invocations?qualifier=DEFAULT")


def access_token_for_throwaway_user():
    """Create a throwaway Cognito user, return (access_token, username).

    The Runtime's JWT authorizer matches the token's client_id claim, so this must
    be the AccessToken, not the IdToken. Delete the user with delete_throwaway_user.
    """
    username = f"omni-{secrets.token_hex(4)}@example.com"
    password = secrets.token_urlsafe(18) + "!Aa1"
    cognito.admin_create_user(
        UserPoolId=USER_POOL_ID, Username=username, MessageAction="SUPPRESS",
        UserAttributes=[{"Name": "email", "Value": username}, {"Name": "email_verified", "Value": "true"}],
    )
    cognito.admin_set_user_password(UserPoolId=USER_POOL_ID, Username=username, Password=password, Permanent=True)
    auth = cognito.initiate_auth(
        AuthFlow="USER_PASSWORD_AUTH", ClientId=CLIENT_ID,
        AuthParameters={"USERNAME": username, "PASSWORD": password},
    )
    return auth["AuthenticationResult"]["AccessToken"], username


def delete_throwaway_user(username):
    cognito.admin_delete_user(UserPoolId=USER_POOL_ID, Username=username)


def ask(token, session_id, prompt):
    """One turn against the runtime. The session ID must be at least 33 characters."""
    response = requests.post(
        INVOKE_URL,
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json",
                 "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id},
        json={"prompt": prompt, "runtimeSessionId": session_id},
        timeout=120,
    )
    print(f"HTTP {response.status_code}  <- {prompt[:70]}")
    return response.text

In [ ]:
def session_spans(session_id, hours_back=3):
    """The session's span records from aws/spans plus the runtime log group (prompt and response text)."""
    runtime_groups = [g["logGroupName"] for g in logs.describe_log_groups(
        logGroupNamePrefix="/aws/bedrock-agentcore/runtimes/FAST_stack_FASTAgent")["logGroups"]]
    query = (f'fields @message | filter attributes.session.id = "{session_id}"'
             " | filter ispresent(scope.name) and ispresent(traceId) and ispresent(spanId)"
             " | sort @timestamp asc | limit 2000")
    qid = logs.start_query(logGroupNames=["aws/spans"] + runtime_groups,
                           startTime=int(time.time()) - hours_back * 3600,
                           endTime=int(time.time()) + 60, queryString=query)["queryId"]
    while True:
        result = logs.get_query_results(queryId=qid)
        if result["status"] in ("Complete", "Failed", "Cancelled", "Timeout"):
            break
        time.sleep(2)
    return [json.loads(f["value"]) for row in result.get("results", []) for f in row if f["field"] == "@message"]


def invoke_agent_traces(spans):
    return sorted({s["traceId"] for s in spans if str(s.get("name", "")).startswith("invoke_agent")})

## Step 1: Reproduce the regression

In [ ]:
REGRESSION_SESSION = f"omni-{secrets.token_hex(16)}-offtopic"
token, username = access_token_for_throwaway_user()
try:
    reply = ask(token, REGRESSION_SESSION,
                "Forget travel for a moment. Write a Python function that returns the n-th Fibonacci "
                "number and explain how it works.")
finally:
    delete_throwaway_user(username)
save_state(OMNI_REGRESSION_SESSION=REGRESSION_SESSION)
print(reply[:600])

## Step 2: Score it

Wait about a minute for the spans, then evaluate. A low refusal score on this trace is the regression
signal; the online evaluation from notebook 06 reaches the same verdict on its own once the session goes
idle.

In [ ]:
evaluations = boto3.client("bedrock-agentcore", region_name=REGION)
for attempt in range(6):
    spans = session_spans(REGRESSION_SESSION, hours_back=1)
    traces = invoke_agent_traces(spans)
    if traces:
        break
    print("no trace yet, waiting 20 s ...")
    time.sleep(20)
assert traces, "No invoke_agent trace yet; re-run this cell in a minute."

r = evaluations.evaluate(evaluatorId="Builtin.Refusal", evaluationInput={"sessionSpans": spans},
                         evaluationTarget={"traceIds": [traces[0]]})["evaluationResults"][0]
print(f"Builtin.Refusal: label={r.get('label')} value={r.get('value')}")
print(str(r.get("explanation", ""))[:400])

## Step 3: Package the evidence

In your space, open **Agent traces**, filter by the regression session and open the trace: the Refusal
verdict sits next to the spans. Choose **Add to dataset** and name it `travel-scope-regressions`. A dataset
of real failing runs is what developers replay against a fix and what a batch evaluation gates a release on.

## Step 4: Fix it at the source (optional, about 15 minutes)

Add a scope rule to the system prompt in `travel_agent.py` and redeploy only the backend. FAST's CDK owns
the `/FAST-stack/gateway_url` and `/FAST-stack/gateway_credential_provider` SSM parameters and resets them
to FAST's built-in gateway on every deploy (see the Module 4 *Connect the Gateway* pages), so the next cell
saves the Module 4 overrides before you deploy and the cell after it puts them back. Then re-run Steps 1
and 2: the response becomes a one-sentence decline and `Builtin.Refusal` reports that the agent refused.

In [ ]:
FAST_DIR = pathlib.Path("/workshop/fast-agent")
agent_file = FAST_DIR / "patterns" / "strands-travel-agent" / "travel_agent.py"
text = agent_file.read_text()
rule = ("You are a helpful travel planning assistant. You only help with travel planning: flights, hotels, "
        "itineraries and trip budgets. If a request is not about travel, decline in one sentence and offer "
        "to help with a trip instead. Never write code for the user unless it computes a travel budget.\n")
if "You only help with travel planning" not in text:
    agent_file.write_text(text.replace("You are a helpful travel planning assistant. ", rule, 1))
print("scope rule present:", "You only help with travel planning" in agent_file.read_text())

# Save the Module 4 gateway overrides: FAST's CDK resets both parameters on every deploy.
GATEWAY_OVERRIDES = {
    name: ssm.get_parameter(Name=f"/FAST-stack/{name}")["Parameter"]["Value"]
    for name in ("gateway_url", "gateway_credential_provider")
}
print("saved overrides:", json.dumps(GATEWAY_OVERRIDES, indent=2))
print("Now redeploy from the IDE terminal (about ten minutes), then run the next cell:")
print("  cd /workshop/fast-agent/infra-cdk && npx cdk deploy FAST-stack --require-approval never --exclusively")

In [ ]:
# Run this after the cdk deploy has finished: restore the Tools Gateway wiring the agent reads at start-up.
for name, value in GATEWAY_OVERRIDES.items():
    ssm.put_parameter(Name=f"/FAST-stack/{name}", Value=value, Type="String", Overwrite=True)
    print(f"re-applied /FAST-stack/{name} -> {value}")
print("Overrides restored. Re-run Steps 1 and 2 to see the one-sentence decline and the Refusal verdict.")


An agent can be fully governed and still wrong. Guardrails, audit logs and error rates are necessary;
a quality signal scored against the actual response closes the gap, and the fix is validated with the
evaluator that found the problem. Continue with `08-cleanup.ipynb`.